In [26]:
import sys
import os
import re
import random
import ollama
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
 
sys.path.append("..")
from utils.preprocessing import LABELS, encode_labels 

DATA_DIR = "../data"
PROBS_DIR = "../results/probs"
RESULTS_DIR = "../results"
os.makedirs(PROBS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
 
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
 
text_col = "sentence" if "sentence" in val_df.columns else "text"
 
label_map = {0: "negative", 1: "neutral", 2: "positive"}
if pd.api.types.is_numeric_dtype(train_df["label"]):
    train_df["label_str"] = train_df["label"].map(label_map)
    val_df["label_str"] = val_df["label"].map(label_map)
else:
    train_df["label_str"] = train_df["label"].astype(str).str.lower().str.strip()
    val_df["label_str"] = val_df["label"].astype(str).str.lower().str.strip()
 
print(f"Loaded {len(train_df)} train samples and {len(val_df)} validation samples.")
print(f"Target classes: {LABELS}")

Loaded 2901 train samples and 967 validation samples.
Target classes: ['negative', 'neutral', 'positive']


In [27]:
MARGIN_THRESHOLD = 0.2
 
baseline_probs_path = os.path.join(PROBS_DIR, "baseline_logreg_val.csv")
assert os.path.exists(baseline_probs_path), f"Missing {baseline_probs_path}!"
 
base_val_df = pd.read_csv(baseline_probs_path)
assert len(base_val_df) == len(val_df), "Probs file and val.csv have different lengths!"
 
val_probs = base_val_df[["p_negative", "p_neutral", "p_positive"]].values
sorted_probs = np.sort(val_probs, axis=1)
margins = sorted_probs[:, -1] - sorted_probs[:, -2]
hard_mask = margins <= MARGIN_THRESHOLD
 
print(f"Validation subset breakdown: {hard_mask.sum()} Hard, {(~hard_mask).sum()} Non-hard "
      f"(expected 153 / 814).")
 
few_shot_examples = []
for label_name in LABELS:
    samples = (
        train_df[train_df["label_str"] == label_name]
        .sample(n=3, random_state=42)[text_col]
        .tolist()
    )
    for s in samples:
        few_shot_examples.append(f'Headline: "{s.strip()}"\nSentiment: {label_name}')
 
random.Random(42).shuffle(few_shot_examples)  
FEW_SHOT_CONTEXT = "\n\n".join(few_shot_examples)
 
SYSTEM_PROMPT = (
    "You are an expert financial sentiment classifier. "
    "Classify the sentiment of the headline into exactly one of three categories: "
    "negative, neutral, or positive. Respond with ONLY the single label word in lowercase."
)
 
MODEL_NAME = "qwen2.5:7b"
 
print("\nConfigured 9-shot prompt context (preview):")
print(FEW_SHOT_CONTEXT[:500] + "\n...")
 
 
def predict_qwen(raw_sentence):
    """Returns 'negative' / 'neutral' / 'positive', or None on failure/unparseable output."""
    user_prompt = (
        f'{FEW_SHOT_CONTEXT}\n\nHeadline: "{raw_sentence.strip()}"\nSentiment:'
    )
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            options={"temperature": 0.0, "seed": 42, "num_predict": 5},
        )
        raw_output = response["message"]["content"].strip().lower()
        match = re.search(r"\b(negative|neutral|positive)\b", raw_output)
        return match.group(1) if match else None
    except Exception as e:
        print(f"Inference error: {e}")
        return None
 
 

print(f"\nRunning inference with {MODEL_NAME} across {len(val_df)} validation samples...")
val_preds = [predict_qwen(s) for s in tqdm(val_df[text_col])]
val_df["qwen_pred"] = val_preds
 
n_bad = val_df["qwen_pred"].isna().sum()
print(f"Unparsed/failed outputs: {n_bad} ({n_bad / len(val_df):.2%})")
val_df["qwen_pred"] = val_df["qwen_pred"].fillna("neutral")  
 
val_pred_export = pd.DataFrame({
    "sentence": val_df[text_col],
    "label": val_df["label_str"],
    "pred": val_df["qwen_pred"],
    "is_hard": hard_mask,
})
val_pred_export.to_csv(os.path.join(RESULTS_DIR, "qwen_val_predictions.csv"), index=False)

Validation subset breakdown: 153 Hard, 814 Non-hard (expected 153 / 814).

Configured 9-shot prompt context (preview):
Headline: "Equipment will be manufactured in Vaahto 's workshop in Hollola , Finland and is scheduled for shipments during the first quarter of 2009 ."
Sentiment: neutral

Headline: "Both loans will be used to finance strategic investments such as shopping center redevelopment projects and refinancing of maturing debt ."
Sentiment: positive

Headline: "Cargo volume grew by 7 % ."
Sentiment: positive

Headline: "The casing comprises a first side casing member provided with the first side vat segm
...

Running inference with qwen2.5:7b across 967 validation samples...


100%|██████████| 967/967 [04:57<00:00,  3.25it/s]

Unparsed/failed outputs: 0 (0.00%)


In [28]:
y_true_all = np.asarray(encode_labels(val_df["label_str"]))
y_pred_all = np.asarray(encode_labels(val_df["qwen_pred"]))
 
y_true_hard, y_pred_hard = y_true_all[hard_mask], y_pred_all[hard_mask]
y_true_easy, y_pred_easy = y_true_all[~hard_mask], y_pred_all[~hard_mask]
 
F1_KW = dict(average="macro", labels=[0, 1, 2], zero_division=0)
 
acc_overall = accuracy_score(y_true_all, y_pred_all)
f1_overall = f1_score(y_true_all, y_pred_all, **F1_KW)
 
acc_hard = accuracy_score(y_true_hard, y_pred_hard)
f1_hard = f1_score(y_true_hard, y_pred_hard, **F1_KW)
 
acc_nonhard = accuracy_score(y_true_easy, y_pred_easy)
f1_nonhard = f1_score(y_true_easy, y_pred_easy, **F1_KW)
 
per_class_f1 = f1_score(y_true_all, y_pred_all, average=None, labels=[0, 1, 2], zero_division=0)
f1_neg, f1_neu, f1_pos = per_class_f1
 
print("=" * 60)
print("QWEN 2.5 EVALUATION RESULTS")
print(f"Overall Accuracy:  {acc_overall:.4f} | Overall Macro-F1:  {f1_overall:.4f}")
print(f"Hard Accuracy:     {acc_hard:.4f} | Hard Macro-F1:     {f1_hard:.4f}")
print(f"Non-Hard Accuracy: {acc_nonhard:.4f} | Non-Hard Macro-F1: {f1_nonhard:.4f}")
print(f"Per-Class F1 -> Negative: {f1_neg:.4f} | Neutral: {f1_neu:.4f} | Positive: {f1_pos:.4f}")
print("=" * 60)
 

metrics_row = pd.DataFrame([{
    "condition": "llm_benchmark",
    "embedding_config": "native_qwen",
    "architecture": "qwen2.5-7b-instruct",
    "accuracy_overall": round(acc_overall, 4),
    "f1_overall": round(f1_overall, 4),
    "accuracy_hard": round(acc_hard, 4),
    "f1_hard": round(f1_hard, 4),
    "accuracy_nonhard": round(acc_nonhard, 4),
    "f1_nonhard": round(f1_nonhard, 4),
    "f1_negative": round(f1_neg, 4),
    "f1_neutral": round(f1_neu, 4),
    "f1_positive": round(f1_pos, 4),
}])
 
metrics_file = os.path.join(RESULTS_DIR, "metrics.csv")
if os.path.exists(metrics_file) and os.path.getsize(metrics_file) > 0:
    metrics_df = pd.read_csv(metrics_file)
    metrics_df = metrics_df[metrics_df["condition"] != "llm_benchmark"]
    updated_metrics = pd.concat([metrics_df, metrics_row], ignore_index=True)
    updated_metrics.to_csv(metrics_file, index=False)
else:
    metrics_row.to_csv(metrics_file, index=False)
 
print(f"\nUpdated benchmark metrics written to {metrics_file}")
 

QWEN 2.5 EVALUATION RESULTS
Overall Accuracy:  0.7766 | Overall Macro-F1:  0.7860
Hard Accuracy:     0.6667 | Hard Macro-F1:     0.6743
Non-Hard Accuracy: 0.7973 | Non-Hard Macro-F1: 0.8085
Per-Class F1 -> Negative: 0.8374 | Neutral: 0.7938 | Positive: 0.7267

Updated benchmark metrics written to ../results/metrics.csv
